In [5]:
# ===================================================
# Raw data : Population-Weighted_Ultraviolet_Irradiance_2004_2015
# Purpose Agreegating OMIUV Data.    
# File Creator : Neha Khairnar
# ==================================================
# 📦 Step 1: Imports
# =========================
import pandas as pd
from pathlib import Path


# =========================
# 📂 Step 2: Load Data
# =========================
BASE_DIR = Path().resolve().parent
#DATA_PATH = BASE_DIR / "data" / "dataset" / "omiuv.csv"   # change if needed
#df = pd.read_csv(DATA_PATH)

#df = pd.read_csv("../data/Datasets/SCHARE_OMIUV_Ultraviolet_Irradiance_2004_2015/Population-Weighted_Ultraviolet_Irradiance_2004_2015.csv", nrows=100)
df = pd.read_csv("../data/Datasets/SCHARE_OMIUV_Ultraviolet_Irradiance_2004_2015/Population-Weighted_Ultraviolet_Irradiance_2004_2015.csv")

print("Raw Data Preview:")
display(df.head(10))


# =========================
# 🧹 Step 3: Rename Columns
# =========================
df.columns = [
    "statefips",
    "countyfips",
    "year",
    "month",
    "day",
    "edd",
    "edr",
    "i305",
    "i310",
    "i324",
    "i380"
]


# =========================
# 📅 Step 4: Create Date Column
# =========================
# below is with standard format YYYY-MM-DD

df["date"] = pd.to_datetime(df[["year", "month", "day"]])

# =========================
# 🧱 Step 5: Fix FIPS Codes
# =========================
# State FIPS → 2 digits
df["state_fips"] = df["statefips"].astype(str).str.zfill(2)

# County FIPS (4-digit given → ensure consistent format)
df["countyfips"] = df["countyfips"].astype(str).str.zfill(5)

# Extract county part (last 3 digits)
df["county_fips"] = df["countyfips"].str[-3:]


# =========================
# 🧾 Step 6: Clean Dataset
# =========================
omiuv_clean = df[
    [
        "year",
        "date",
        "state_fips",
        "county_fips",
        "edd",
        "edr",
        "i305",
        "i310",
        "i324",
        "i380"
    ]
]

print("Cleaned OMIUV Data:")
display(omiuv_clean.head(10))

# ======================================================================
# 💾 Step 8: Save Clean Dataset
# =======================================================================

OUTPUT_PATH = BASE_DIR / "data" / "processed" / "neighbourhood_omiuv_clean.csv"

# Create folder if not exists
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
omiuv_clean.to_csv(OUTPUT_PATH, index=False)
print(f"\n✅ File saved to: {OUTPUT_PATH}")


# =========================
# 📊 Step 7: Yearly Aggregation
# =========================
omiuv_yearly = (
    omiuv_clean.groupby(["year", "state_fips", "county_fips"])
    .agg(
        EDD_mean=("edd", "mean"),
        EDR_mean=("edr", "mean"),
        I305_mean=("i305", "mean"),
        I310_mean=("i310", "mean"),
        I324_mean=("i324", "mean"),
        I380_mean=("i380", "mean")
    )
    .reset_index()
)


# =========================
# 👀 Step 8: Preview Yearly Data
# =========================
print("Yearly OMIUV Dataset:")
display(omiuv_yearly.head(10))


# =========================
# 📊 Step 9: Validate
# =========================
print("Shape:", omiuv_yearly.shape)
print(omiuv_yearly.info())

# =========================
# 💾 Step 10: Save
# =========================
OUTPUT_PATH = BASE_DIR / "data" / "processed" / "neighbourhood_omiuv_yearly.csv"

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

omiuv_yearly.to_csv(OUTPUT_PATH, index=False)

print("Saved to:", OUTPUT_PATH)


Raw Data Preview:


,statefips,countyfips,year,month,day,edd,edr,i305,i310,i324,i380
0,1,1001,2004,10,1,3340.371010,173.484822,49.325671,85.828488,335.582135,633.891882
1,1,1101,2004,10,1,3136.522793,163.955051,46.690414,81.002302,315.412775,591.526199
2,1,1103,2004,10,1,3280.537044,169.355457,47.694926,84.284288,334.736750,636.723289
3,1,1105,2004,10,1,3259.015568,169.805809,48.351506,83.936939,327.325647,616.582604
4,1,1107,2004,10,1,3194.306285,166.347097,47.094928,82.524411,324.712665,612.686963
5,1,1109,2004,10,1,3027.287774,159.068669,45.561733,78.473303,302.879586,561.660874
6,1,1011,2004,10,1,3060.379510,160.438121,45.850892,79.160829,306.528969,571.719766
7,1,1111,2004,10,1,3285.775716,170.412810,48.064103,84.733321,335.222971,633.025597
8,1,1113,2004,10,1,3087.365619,161.474225,45.876161,79.901522,312.125982,584.444605
9,1,1115,2004,10,1,3373.667306,174.037352,49.067870,86.517856,342.742813,650.292814


Cleaned OMIUV Data:


,year,date,state_fips,county_fips,edd,edr,i305,i310,i324,i380
0,2004,2004-10-01,01,001,3340.371010,173.484822,49.325671,85.828488,335.582135,633.891882
1,2004,2004-10-01,01,101,3136.522793,163.955051,46.690414,81.002302,315.412775,591.526199
2,2004,2004-10-01,01,103,3280.537044,169.355457,47.694926,84.284288,334.736750,636.723289
3,2004,2004-10-01,01,105,3259.015568,169.805809,48.351506,83.936939,327.325647,616.582604
4,2004,2004-10-01,01,107,3194.306285,166.347097,47.094928,82.524411,324.712665,612.686963
5,2004,2004-10-01,01,109,3027.287774,159.068669,45.561733,78.473303,302.879586,561.660874
6,2004,2004-10-01,01,011,3060.379510,160.438121,45.850892,79.160829,306.528969,571.719766
7,2004,2004-10-01,01,111,3285.775716,170.412810,48.064103,84.733321,335.222971,633.025597
8,2004,2004-10-01,01,113,3087.365619,161.474225,45.876161,79.901522,312.125982,584.444605
9,2004,2004-10-01,01,115,3373.667306,174.037352,49.067870,86.517856,342.742813,650.292814



✅ File saved to: C:\Users\NVK00\Desktop\MyData\Projects\Clinical_Neighbourhood_ML_Data_Predictor\data\processed\neighbourhood_omiuv_clean.csv
Yearly OMIUV Dataset:


,year,state_fips,county_fips,EDD_mean,EDR_mean,I305_mean,I310_mean,I324_mean,I380_mean
0,2004,01,001,1635.143311,90.813744,23.443177,46.061823,204.497604,389.849516
1,2004,01,003,1854.235452,102.445959,27.227998,51.497416,220.103421,416.603439
2,2004,01,005,1758.538366,97.520301,25.462974,49.366290,216.160133,412.010815
3,2004,01,007,1533.330234,85.366267,21.740807,43.416436,196.066400,374.993541
4,2004,01,009,1439.734620,80.153622,20.128070,40.845879,187.658463,359.458481
5,2004,01,011,1718.277413,95.413782,24.789638,48.348187,213.105039,406.612612
6,2004,01,013,1692.378700,93.905892,24.493337,47.567195,208.623713,396.966700
7,2004,01,015,1506.274908,83.801683,21.171422,42.692830,194.839906,373.199838
8,2004,01,017,1628.409048,90.580053,23.239365,46.028322,206.102422,393.690137
9,2004,01,019,1473.230889,81.983531,20.581418,41.791712,192.256721,369.041198


Shape: (37308, 9)
<class 'pandas.DataFrame'>
RangeIndex: 37308 entries, 0 to 37307
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   year         37308 non-null  int64  
 1   state_fips   37308 non-null  str    
 2   county_fips  37308 non-null  str    
 3   EDD_mean     37308 non-null  float64
 4   EDR_mean     37308 non-null  float64
 5   I305_mean    37308 non-null  float64
 6   I310_mean    37308 non-null  float64
 7   I324_mean    37308 non-null  float64
 8   I380_mean    37308 non-null  float64
dtypes: float64(6), int64(1), str(2)
memory usage: 2.6 MB
None
Saved to: C:\Users\NVK00\Desktop\MyData\Projects\Clinical_Neighbourhood_ML_Data_Predictor\data\processed\neighbourhood_omiuv_yearly.csv
